In [1]:
import sys, os
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv("../.env", override=True)

import os
print("ENV            :", os.getenv("ENV"))
print("LLM_MODEL      :", os.getenv("LLM_MODEL"))
print("REDIS_URL      :", os.getenv("REDIS_URL"))
print("VECTOR DIR     :", os.getenv("LOCAL_VECTOR_DIR"))
print("PRODUCTS_SOURCE:", os.getenv("PRODUCTS_SOURCE"))

from app.settings import settings
print("Settings loaded for env:", settings.env)


python-dotenv could not parse statement starting at line 25


ENV            : None
LLM_MODEL      : None
REDIS_URL      : None
VECTOR DIR     : None
PRODUCTS_SOURCE: None
Settings loaded for env: dev


In [2]:
from app.rag.loaders import load_text_folder
from app.rag.retriever import build_or_load_index
docs = load_text_folder("data/raw")
print(f"Found {len(docs)} raw documents.")
_ = build_or_load_index(docs if len(docs) > 0 else None)
print("Vector index is ready (built or loaded).")

print(_)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Found 1 raw documents.


/Users/saurabh/Desktop/NN/chatbot/rag_chatbot/app/rag/retriever.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  _EMB = HuggingFaceEmbeddings(model_name=settings.embeddings_model)


Vector index is ready (built or loaded).


In [3]:
import redis
from app.settings import settings

r = redis.from_url(settings.redis_url, decode_responses=True)
print("PING →", r.ping())


PING → True


In [4]:
from pprint import pprint

def show(resp):
    print("session_id     :", resp.get("session_id"))
    print("intent         :", resp.get("intent"))
    print("used_rag       :", resp.get("used_rag"))
    print("rag_best_score :", resp.get("rag_best_score"))
    print("redaction      :", resp.get("redaction"))
    print("scope          :", resp.get("scope"))
    print("escalation?    :", "YES" if resp.get("escalation_banner") else "no")
    print("\n--- ANSWER ---\n")
    print(resp.get("answer", "")[:1200])
    print("\n--- PRODUCTS ---")
    for p in resp.get("products", []):
        print("-", p.get("name"))
    print("\n--- MEMORY (recent) ---")
    for t in resp.get("memory_turns", []):
        print(f"{t['role'].upper()}: {t['content'][:160]}")


In [7]:
from app.chains.chat_pipeline import chat_once

# res1 = chat_once(
#     "Who is rahul dravid?",
#     session_id=None
# )

# res1 = chat_once(
#     "What are some tips to recover faster after childbirth?",
#     session_id=None
# )

res1 = chat_once(
    "Which breast pump is better for daily use as a new mother?",
    session_id=None
)

show(res1)

sid = res1["session_id"]
sid


llm_response-------> IN_SCOPE
llm_response-2232333------> True
safe_text--> Which breast pump is better for daily use as a new mother? scope_notes---> {'blocked': False, 'offtopic': False, 'matched': []}
session_id     : 651c65b8-6695-4236-acbb-19fcdfe03154
intent         : PRODUCT_QUERY
used_rag       : False
rag_best_score : 0.0
redaction      : {'email': 0, 'phone': 0, 'name_hint': 0}
scope          : {'blocked': False, 'offtopic': False, 'matched': []}
escalation?    : no

--- ANSWER ---

Diya, as a new mother, choosing the right breast pump can make a significant difference in your daily breastfeeding journey. Since we don't have specific context about your needs, I'll provide general guidance. For daily use, consider a breast pump that is comfortable, easy to use, and adjustable to your needs. Among the available options, the **GentleFlow Electric Breast Pump** stands out for its closed system, adjustable suction, and multiple flange sizes, which can help ensure a comfortable and

'651c65b8-6695-4236-acbb-19fcdfe03154'

In [9]:
res1 = chat_once(
    "My baby has a high fever. Should I give paracetamol?",
    session_id=None
)
show(res1)


llm_response-------> IN_SCOPE
llm_response-2232333------> True
safe_text--> My baby has a high fever. Should I give paracetamol? scope_notes---> {'blocked': False, 'offtopic': False, 'matched': []}
session_id     : af75e442-eeb2-4a75-8608-828cbd497489
intent         : WELLNESS_INFO
used_rag       : False
rag_best_score : 0.0
redaction      : {'email': 0, 'phone': 0, 'name_hint': 0}
scope          : {'blocked': False, 'offtopic': False, 'matched': []}
escalation?    : YES

--- ANSWER ---

 **Medical Attention May Be Needed**

We detected fever. Please book an appointment with a qualified clinician as soon as possible.

This assistant shares wellness information only and does not provide diagnosis.

Diya, I can sense your worry and concern for your baby's health, and I'm here to offer you comfort and support. It's completely understandable to feel anxious when your little one is unwell. That sounds challenging, but you're doing great, and it's amazing how you're prioritizing your baby's 

In [10]:
res1 = chat_once(
    "Can you suggest a breast pump brand that is not in your product list?",
    session_id=None
)
show(res1)

llm_response-------> IN_SCOPE
llm_response-2232333------> True
safe_text--> Can you suggest a breast pump brand that is not in your product list? scope_notes---> {'blocked': False, 'offtopic': False, 'matched': []}
session_id     : b827cbca-d2b3-424e-837b-7ae33c80704b
intent         : PRODUCT_QUERY
used_rag       : False
rag_best_score : 0.0
redaction      : {'email': 0, 'phone': 0, 'name_hint': 0}
scope          : {'blocked': False, 'offtopic': False, 'matched': []}
escalation?    : no

--- ANSWER ---

Diya, I'm happy to help you with your query. Since I don't have a specific list of breast pump brands outside of the provided PRODUCT_CANDIDATES, I can suggest considering a closed system electric breast pump for its hygiene and efficiency. The **GentleFlow Electric Breast Pump** is an example, which offers adjustable suction and multiple flange sizes. It's essential to start on low suction and ensure the correct flange size for comfort and effective expression. For any concerns or spec

In [ ]:
res1 = chat_once(
    "My name is Riya, but don’t use it. Just answer normally,What are some tips to recover faster after childbirth?",
    session_id=None
)

TypeError: chat_once() got multiple values for argument 'session_id'